#### PyTorch Fundamentals

Core data structure of PyTorch is a tensor. A multidimensional array with a shape and a data type. Kind of like an array, but with support for GPU and support auto-differentiation.

Creation of a tensor

In [2]:
import torch

# let's create a 2 x 3 array (tensor)
X = torch.tensor([[1, 2, 3], [4, 5, 6]])
X

tensor([[1, 2, 3],
        [4, 5, 6]])

Info

In [3]:
X.shape

torch.Size([2, 3])

In [4]:
X.dtype

torch.int64

Indexing

In [5]:
X[0, 1]

tensor(2)

In [6]:
X[:, 1]

tensor([2, 5])

#### Hardware Acceleration

In [7]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

'cuda'

Now let's create a tensor on that GPU

In [8]:
M = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
M = M.to(device)

In [9]:
M.device

device(type='cuda', index=0)

In [10]:
M = torch.tensor([[1., 2., 3.], [4., 5., 6.]], device=device)

Run something on the gpu

In [11]:
R = M @ M.T
R

tensor([[14., 32.],
        [32., 77.]], device='cuda:0')

In [12]:
M = torch.rand((1000, 1000))  # on the CPU
%timeit M @ M.T

6.32 ms ± 62.9 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [13]:
M = torch.rand((1000, 1000), device="cuda")  # on the GPU
%timeit M @ M.T

224 μs ± 804 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


#### Autograd

PyTorch comes with an efficient implementation of reverse-mode auto-differentiation, called `autograd`, which stands for `automatic gradients`.

Consider a simple function, f(x) = x^2. Derivitive of this is f'(x) = 2x so eval on f(x) and f'(x) should be 25 and 10.

In [14]:
x = torch.tensor(5.0, requires_grad=True)
f = x ** 2
f

tensor(25., grad_fn=<PowBackward0>)

In [15]:
f.backward()
x.grad

tensor(10.)

After computing the gradients, you generally want to perform a gradien descent step by subtracting a fraction of the gradients from the model variables.

In [16]:
learning_rate = 0.1
with torch.no_grad():
    x -= learning_rate * x.grad

or use `detach()` which is detached from the computational graph.

In [17]:
x_detached = x.detach()
x_detached -= learning_rate * x.grad

Before repeating the process (forward pass + backward pass + gradient descent step), it is essential to zero out the gradients of every model params.

In [18]:
x.grad.zero_()

tensor(0.)

Putting it all together, this how the training loop looks like:

In [19]:
learning_rate = 0.1
x = torch.tensor(5.0, requires_grad=True)
for iteration in range(100):
    f = x ** 2 # forward pass
    f.backward() # backward pass
    with torch.no_grad():
        x -= learning_rate * x.grad # gradient descent step

    x.grad.zero_()

#### Implementing Linear Regression

In [20]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

housing = fetch_california_housing()
X_train_full, X_test, y_train_full, y_test = train_test_split(
    housing.data, housing.target, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, random_state=42)

Let's cast our dataset to FloatTensor

In [21]:
X_train = torch.FloatTensor(X_train)
X_valid = torch.FloatTensor(X_valid)
X_test = torch.FloatTensor(X_test)
means = X_train.mean(dim=0, keepdims=True)
stds = X_train.std(dim=0, keepdims=True)
X_train = (X_train - means) / stds
X_valid = (X_valid - means) / stds
X_test = (X_test - means) / stds

And convert our targets as well

In [22]:
y_train = torch.FloatTensor(y_train).reshape(-1, 1)
y_valid = torch.FloatTensor(y_valid).reshape(-1, 1)
y_test = torch.FloatTensor(y_test).reshape(-1, 1)

Let's create the params for our model

In [23]:
torch.manual_seed(42)
n_features = X_train.shape[1] # there are 8 input features
w = torch.rand((n_features, 1), requires_grad=True) # weights
b = torch.tensor(0., requires_grad=True) # bias

Next, let's train our model.

In [24]:
learning_rate = 0.4
n_epochs = 20
for epoch in range(n_epochs):
    y_pred = X_train @ w + b
    loss = ((y_pred - y_train) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
        w.grad.zero_()
        b.grad.zero_()
    print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item()}")

Epoch 1/20, Loss: 7.792580604553223
Epoch 2/20, Loss: 1.3588199615478516
Epoch 3/20, Loss: 0.84256911277771
Epoch 4/20, Loss: 0.7366544604301453
Epoch 5/20, Loss: 0.6970381140708923
Epoch 6/20, Loss: 0.6746658086776733
Epoch 7/20, Loss: 0.6581473350524902
Epoch 8/20, Loss: 0.6443582773208618
Epoch 9/20, Loss: 0.6323144435882568
Epoch 10/20, Loss: 0.6216264367103577
Epoch 11/20, Loss: 0.6120853424072266
Epoch 12/20, Loss: 0.6035464406013489
Epoch 13/20, Loss: 0.5958940982818604
Epoch 14/20, Loss: 0.5890297889709473
Epoch 15/20, Loss: 0.5828677415847778
Epoch 16/20, Loss: 0.577332079410553
Epoch 17/20, Loss: 0.5723556280136108
Epoch 18/20, Loss: 0.5678789019584656
Epoch 19/20, Loss: 0.5638489723205566
Epoch 20/20, Loss: 0.5602186918258667


Now we can predict

In [25]:
X_new = X_test[:3] # pretend these are new instances
with torch.no_grad():
    y_pred = X_new @ w + b # use the trained params to make predictions
y_pred

tensor([[0.8684],
        [1.6626],
        [2.7272]])

Now let's rewrite using higher-level API.